# **Teknik Scraping Website**

## Alat

Berikut ini adalah library yang dibutuhkan untuk mengambil data berita pada detik.com

1. requests
2. beautifulSoap
3. trafilatura
4. pandas

Proyek ini berfokus pada teknik ekstraksi data (web scraping) untuk membangun dataset teks berbahasa Indonesia dari portal berita daring. Melalui proses otomatisasi, ribuan tautan dari kategori spesifik dikumpulkan secara terstruktur. Data mentah yang dihasilkan difilter dari elemen visual pengganggu (seperti iklan dan menu navigasi) untuk mendapatkan murni teks artikel. Dataset akhir yang tersimpan dalam format tabular CSV ini sangat ideal untuk mempermudah tahap prapemrosesan teks, seperti tokenization dan ekstraksi fitur, sebelum digunakan lebih lanjut dalam pengembangan model klasifikasi teks.

Peran Masing-Masing Alat

* Requests: Menangani komunikasi klien-server (HTTP/HTTPS) untuk mengunduh dokumen HTML secara mentah.

* BeautifulSoup4: Membedah struktur HTML untuk mengisolasi dan mengumpulkan tautan berita yang relevan berdasarkan penyeleksi CSS.

* Trafilatura: Algoritma ekstraksi cerdas yang memisahkan teks utama artikel dari elemen non-konten (HTML boilerplate).

* Pandas: Mengonversi kumpulan teks dan URL yang terisolasi menjadi struktur tabel (kolom dan baris) standar industri untuk diekspor menjadi file CSV.

In [2]:
import requests
from bs4 import BeautifulSoup
import trafilatura
import pandas as pd
import time
from tqdm import tqdm

### Fungsi mendapatkan URL dalam Website Detik.com

In [3]:
def dapatkan_url_detik(url_dasar, target_jumlah=100):
    """Mengumpulkan URL artikel dengan format pagination query (?page=)."""
    url_list = []
    halaman = 1
    
    print(f"Mulai mencari di: {url_dasar}")
    
    with tqdm(total=target_jumlah, desc="Mengumpulkan URL") as pbar:
        while len(url_list) < target_jumlah:
            
            # PERBAIKAN FINAL: Menggunakan format query parameter ?page=
            url_indeks = f"{url_dasar}indeks?page={halaman}"
            
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
            }
            
            try:
                response = requests.get(url_indeks, headers=headers, timeout=15)
                
                if response.status_code != 200:
                    print(f"\n[Peringatan] Server menolak akses di {url_indeks} (Status: {response.status_code}).")
                    break
                    
                soup = BeautifulSoup(response.text, 'html.parser')
                
                artikel_elemen = soup.find_all('article')
                
                if not artikel_elemen:
                    print(f"\n[Peringatan] Tidak menemukan tag <article> di halaman {halaman}. Berhenti.")
                    break
                    
                for artikel in artikel_elemen:
                    link = None
                    
                    if 'i-link' in artikel.attrs:
                        link = artikel['i-link']
                    else:
                        tag_a = artikel.find('a')
                        if tag_a and 'href' in tag_a.attrs:
                            link = tag_a['href']
                    
                    # Validasi link
                    if link and url_dasar in link and 'foto' not in link and 'video' not in link:
                        if link not in url_list:
                            url_list.append(link)
                            pbar.update(1)
                            
                            if len(url_list) >= target_jumlah:
                                break
            except Exception as e:
                print(f"\n[Error] Gagal mengakses {url_indeks}: {e}")
                break
                
            halaman += 1
            if halaman > 50:
                print("\n[Batas] Mencapai batas maksimal halaman.")
                break
                
            time.sleep(1) 
            
    return url_list

### Fungsi untuk mengekstrak kata dalam Berita yang sudah di scarping

In [4]:
def ekstrak_teks_trafilatura(url_list, kategori):
    """Mengekstrak teks utama menggunakan Trafilatura dengan progress bar."""
    data_ekstrak = []
    
    # Membungkus url_list dengan tqdm agar muncul progress bar saat ekstraksi
    for url in tqdm(url_list, desc=f"Ekstraksi teks {kategori}"):
        downloaded = trafilatura.fetch_url(url)
        if downloaded:
            teks_bersih = trafilatura.extract(downloaded)
            if teks_bersih:
                data_ekstrak.append({
                    'kategori': kategori,
                    'url': url,
                    'teks': teks_bersih
                })
        time.sleep(1.5)
        
    return data_ekstrak

In [5]:
url_tema_sport = "https://sport.detik.com/"
url_tema_finance = "https://finance.detik.com/"

In [49]:
url_sport = dapatkan_url_detik(url_tema_sport, 100)
url_finance = dapatkan_url_detik(url_tema_finance, 100)

Mulai mencari di: https://sport.detik.com/


Mengumpulkan URL: 100%|██████████| 100/100 [00:10<00:00,  9.99it/s]


Mulai mencari di: https://finance.detik.com/


Mengumpulkan URL: 100%|██████████| 100/100 [00:11<00:00,  8.86it/s]


In [ ]:
data_sport = ekstrak_teks_trafilatura(url_sport, 'sport')
data_finance = ekstrak_teks_trafilatura(url_finance, 'finance')

In [ ]:
# 4. Simpan ke CSV MASING-MASING TEMA SECARA TERPISAH
# Menyimpan Dataset Sport
df_sport = pd.DataFrame(data_sport)
nama_file_sport = 'dataset_detik_sport.csv'
df_sport.to_csv(nama_file_sport, index=False, encoding='utf-8')
print(f"\nDataset Sport berhasil disimpan sebagai: {nama_file_sport}")

# Menyimpan Dataset Finance
df_finance = pd.DataFrame(data_finance)
nama_file_finance = 'dataset_detik_finance.csv'
df_finance.to_csv(nama_file_finance, index=False, encoding='utf-8')
print(f"Dataset Finance berhasil disimpan sebagai: {nama_file_finance}")

## **Data Understanding**

Proses web scraping telah berhasil dieksekusi secara otomatis dan mengumpulkan total 200 data artikel secara berimbang, yakni 100 artikel dari kategori Olahraga (Sport) dan 100 artikel dari kategori Keuangan (Finance). Data mentah tersebut kemudian diekstrak dan direstrukturisasi ke dalam format dataframe tabular dengan tiga atribut (kolom) utama:

* Kategori: Berfungsi sebagai label kelas target dari artikel (sport atau finance).

* URL: Tautan referensi (hyperlink) asli untuk pelacakan sumber berita.

* Teks: Inti konten berita yang telah dibersihkan (clean text) dari elemen navigasi web, iklan, dan boilerplate HTML.

Keberhasilan ekstraksi teks bersih ini sangat krusial. Dengan struktur yang terstandarisasi, dataset kini berada dalam kondisi siap pakai untuk memasuki fase prapemrosesan lanjutan (text preprocessing). Data ini dapat langsung diproses melalui tahapan pembersihan simbol, tokenisasi, hingga ekstraksi fitur (seperti menggunakan 5-fold cross-validation) guna melatih model machine learning klasifikasi sentimen atau teks yang memanfaatkan arsitektur Transformer, seperti model bahasa IndoBERT.

In [8]:
df_sport = pd.read_csv("dataset_detik_sport.csv")
df_finance = pd.read_csv("dataset_detik_finance.csv")

In [9]:
# ==========================================
# INFO DATASET SPORT
# ==========================================
print("=== DATASET SPORT ===")

# 1. Mendapatkan jumlah data (baris) dan kolom
baris_sport, kolom_sport = df_sport.shape
print(f"Jumlah data  : {baris_sport} baris")
print(f"Jumlah kolom : {kolom_sport} kolom\n")

# 2. Menampilkan 3 data teratas
print("Tampilan 3 data teratas:")
display(df_sport.head(3))


=== DATASET SPORT ===
Jumlah data  : 100 baris
Jumlah kolom : 3 kolom

Tampilan 3 data teratas:


,kategori,url,teks
0,sport,https://sport.detik.com/sport-lain/d-8651374/w...,Sean Gelael dan Team WRT 32 akan coba meraih p...
1,sport,https://sport.detik.com/moto-gp/d-8651046/ktm-...,KTM punya sejumlah alasan merekrut Luca Marini...
2,sport,https://sport.detik.com/moto-gp/d-8650852/jadw...,MotoGP 2026 akan berlanjut ke San Marino. Seri...


In [10]:
# ==========================================
# INFO DATASET FINANCE
# ==========================================
print("=== DATASET FINANCE ===")

# 1. Mendapatkan jumlah data (baris) dan kolom
baris_finance, kolom_finance = df_finance.shape
print(f"Jumlah data  : {baris_finance} baris")
print(f"Jumlah kolom : {kolom_finance} kolom\n")

# 2. Menampilkan 3 data teratas
print("Tampilan 3 data teratas:")
display(df_finance.head(3))

=== DATASET FINANCE ===
Jumlah data  : 100 baris
Jumlah kolom : 3 kolom

Tampilan 3 data teratas:


,kategori,url,teks
0,finance,https://finance.detik.com/berita-ekonomi-bisni...,Menteri Perhubungan Dudy Purwagandhi menyataka...
1,finance,https://finance.detik.com/berita-ekonomi-bisni...,Ketua Badan Perlindungan Konsumen Nasional (BP...
2,finance,https://finance.detik.com/berita-ekonomi-bisni...,Produk lokal asal Indonesia kini banyak yang d...


In [11]:
# Menggabungkan kedua dataset (100 sport + 100 finance = 200 data)
df_all = pd.concat([df_sport, df_finance], ignore_index=True)

In [12]:
# ==========================================
# 1. Menghitung Jumlah Kata per Dokumen
# ==========================================
# Membuat kolom baru 'jumlah_kata' dengan memecah teks berdasarkan spasi
df_all['jumlah_kata'] = df_all['teks'].astype(str).apply(lambda x: len(x.split()))

# Menghitung total keseluruhan kata dari ke-200 dokumen
total_kata_keseluruhan = df_all['jumlah_kata'].sum()

In [ ]:
# ==========================================
# 2. Menghitung Jumlah Kata Unik (Vocabulary)
# ==========================================
semua_teks = " ".join(df_all['teks'].astype(str).tolist()).lower()

# Memecah paragraf raksasa menjadi daftar (list) per kata
kumpulan_kata = semua_teks.split()

# Mengubah list menjadi Set (karena Set otomatis membuang data yang duplikat)
kata_unik = set(kumpulan_kata)
jumlah_kata_unik = len(kata_unik)

In [14]:
# ==========================================
# TAMPILKAN HASIL
# ==========================================
print("=== STATISTIK KORPUS TEKS ===")
print(f"Total seluruh dokumen      : {len(df_all)} artikel")
print(f"Total seluruh kata         : {total_kata_keseluruhan:,} kata")
print(f"Total kata UNIK (Vocab)    : {jumlah_kata_unik:,} kata")

print("\n=== SAMPEL JUMLAH KATA PER DOKUMEN ===")
display(df_all[['kategori', 'jumlah_kata', 'teks']].head())

=== STATISTIK KORPUS TEKS ===
Total seluruh dokumen      : 200 artikel
Total seluruh kata         : 67,507 kata
Total kata UNIK (Vocab)    : 11,604 kata

=== SAMPEL JUMLAH KATA PER DOKUMEN ===


,kategori,jumlah_kata,teks
0,sport,182,Sean Gelael dan Team WRT 32 akan coba meraih p...
1,sport,335,KTM punya sejumlah alasan merekrut Luca Marini...
2,sport,636,MotoGP 2026 akan berlanjut ke San Marino. Seri...
3,sport,334,Cal Crutchlow membuat prediksi menarik soal ka...
4,sport,343,Team WRT 32 sudah menuntaskan sesi latihan beb...


In [16]:
print("\n=== SAMPEL KATA UNIK (VOCAB) ===")
print(list(kata_unik))  # Menampilkan 10 kata unik pertama


=== SAMPEL KATA UNIK (VOCAB) ===
['kehidupan', 'terpatri', 'asia.', 'mungkin', 'hiburan,', 'kecil.', 'contender', 'safari,', 'juga', '20.000', 'karbon,', 'partner,', 'banyaknya', '879.920', 'substitusi', 'itdc,', 'dibina.', 'officer', 'memadukan', 'rp350', 'gong', 'tuhan.', 'berdua', 'f&b)', 'berkelanjutan.transformasi', 'menyongsong', 'jargon-jargon', 'kebutuhan,', 'disiagakan', '"refund', 'subsidi', 'kejuaraan.', 'dijalani', 'transaction', '07.00', 'sebelas', 'tujuh', 'keterpurukannya', 'jicc,', 'finis.', 'fortesport,', 'puas.', '"pertama', '35.000', 'menang,"', '"keuntungan', 'regional', 'artifisial,', 'trader', '93,6%.', 'serikat', 'hiu', 'safety', 'kabid', 'sangat', 'p15,', '9-6.', 'lahan.', 'lukman.', 'pppk', 'mari', 'zwinkle,', 'praktis,', 'tertekan', 'deaf', 'validasi,', 'speed', 'mendorong', 'bulog', 'activity', 'ke-22', 'migas', 'alti', 'bangsa,', 'dingin', 'kick.', 'ketika', 'menitipkan', '277', 'dilepaskannya,', 'silverstone,"', 'persatuan', 'rizal', 'mengirimkan', 'impact